# SupplyMind AI — Preprocessing

## Objective

Demonstrate the transformations applied immediately before model fitting:
chronological splitting, stateful feature engineering, one-hot encoding,
numerical imputation/scaling, and protection against unseen categories.

In [1]:
# -------------------
# Imports
# -------------------

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from supplymind.features.predictions.domain.constants import (
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
    TARGET_COLUMN,
)
from supplymind.features.predictions.ml.workflow import load_clean_syndelay

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [2]:
# -------------------
# Project configuration
# -------------------

DATASET_PATH = Path("../data/raw/syndelay/syndelay_v1.csv")
REPORT_ROOT = Path("../reports")

assert DATASET_PATH.exists(), (
    f"Dataset not found: {DATASET_PATH}"
)

In [3]:
from supplymind.features.predictions.ml.preprocessing import build_preprocessor
from supplymind.features.predictions.ml.workflow import (
    load_clean_syndelay,
    prepare_model_data,
)

df = load_clean_syndelay(DATASET_PATH)
data = prepare_model_data(df)

## 1. Chronological split

In [4]:
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [
        len(data.X_train),
        len(data.X_validation),
        len(data.X_test),
    ],
    "positive_rate": [
        data.y_train.mean(),
        data.y_validation.mean(),
        data.y_test.mean(),
    ],
})

split_summary

,split,rows,positive_rate
0,train,108841,0.576465
1,validation,23323,0.577198
2,test,23324,0.578974


### Analysis

The split preserves temporal order:

- earliest 70% → training,
- next 15% → validation,
- latest 15% → test.

This is stricter than the random split used in the reference notebook and
better represents future production scoring.

## 2. One-hot encoding and numerical scaling

In [5]:
preprocessor = build_preprocessor(
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    scale_numerical=True,
)

transformed = preprocessor.fit_transform(data.X_train)

print("Raw train shape:", data.X_train.shape)
print("Encoded train shape:", transformed.shape)

ValueError: Some column names are not columns of the dataframe: {'customer_city_frequency', 'order_month', 'order_is_weekend', 'order_state_frequency', 'order_quarter', 'order_year', 'order_day', 'order_hour', 'order_weekday', 'order_week', 'order_city_frequency'}

### Analysis

**Numerical pipeline**

median imputation → standard scaling

**Categorical pipeline**

most-frequent imputation → OneHotEncoder(handle_unknown="ignore")

One-hot encoding is appropriate for nominal categories such as payment type,
market, product category, and shipping mode because no artificial numeric order
should be imposed.

## 3. Encoded feature names

In [ ]:
feature_names = preprocessor.get_feature_names_out()

print("Total transformed features:", len(feature_names))
pd.DataFrame({"feature": feature_names}).head(100)

### Analysis

The number of transformed columns is larger than the raw model contract because
each categorical value becomes its own binary indicator.

This exact fitted transformer is persisted with the estimator so FastAPI does
not duplicate encoding logic.